## **Dummy example**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import ApproxInference, DBNInference
from pgmpy.models import DynamicBayesianNetwork as DBN

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

/home/camilo/Repositorios/pgmpy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = np.random.choice(["s1", "s2"], size=(1000, 6))
colnames = []
for t in range(2):
    colnames.extend([("A", t), ("B", t), ("C", t)])

df = pd.DataFrame(data, columns=colnames)
df

,"(A, 0)","(B, 0)","(C, 0)","(A, 1)","(B, 1)","(C, 1)"
0,s2,s1,s1,s1,s2,s2
1,s2,s1,s1,s2,s2,s2
2,s1,s2,s1,s1,s1,s1
3,s2,s1,s2,s1,s1,s1
4,s1,s1,s2,s1,s1,s2
...,...,...,...,...,...,...
995,s1,s1,s1,s1,s2,s1
996,s1,s1,s1,s1,s1,s1
997,s2,s2,s1,s1,s1,s1
998,s2,s2,s1,s1,s2,s2


In [3]:
model = DBN(
    [
        (("A", 0), ("B", 0)),
        (("A", 0), ("C", 0)),
        (("A", 0), ("A", 1)),
        (("B", 0), ("B", 1)),
        (("C", 0), ("C", 1)),
    ]
)
print(model._nodes())

['B', 'C', 'A']


In [4]:
model.fit(df)

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'B_0': 'C', 'C_0': 'C', 'A_0': 'C', 'B_1': 'C', 'C_1': 'C', 'A_1': 'C'}


In [5]:
# Query the model
dbn_inf = DBNInference(model)
print(dbn_inf.query([("B", 0)], evidence={("A", 0): "s1"})[("B", 0)])

+-------------+-----------------+
| ('B', 0)    |   phi(('B', 0)) |
+=============+=================+
| ('B', 0)(0) |          0.5078 |
+-------------+-----------------+
| ('B', 0)(1) |          0.4922 |
+-------------+-----------------+


In [6]:
# Query the model
dbn_inf = DBNInference(model)
print(dbn_inf.query([("A", 0)], evidence={("B", 0): "s1", ("C", 0): "s2"})[("A", 0)])

+-------------+-----------------+
| ('A', 0)    |   phi(('A', 0)) |
+=============+=================+
| ('A', 0)(0) |          0.5296 |
+-------------+-----------------+
| ('A', 0)(1) |          0.4704 |
+-------------+-----------------+


In [7]:
# Query the model
dbn_inf = ApproxInference(model)
print(
    dbn_inf.query(
        [("A", 0)],
        evidence={("B", 0): "s1", ("C", 0): "s2"},
        # state_names={("A", 0): ["s1", "s2"]},
    )
)

  0%|          | 0/3 [00:00<?, ?it/s]

+--------------+-----------------+
| ('A', 0)     |   phi(('A', 0)) |
+==============+=================+
| ('A', 0)(s2) |          0.4648 |
+--------------+-----------------+
| ('A', 0)(s1) |          0.5352 |
+--------------+-----------------+



/home/camilo/Repositorios/pgmpy/pgmpy/inference/ApproxInference.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  samples.groupby(variables).size() / samples.shape[0], state_names


In [8]:
# Query the model
dbn_inf = ApproxInference(model)
print(
    dbn_inf.query(
        [("A", 0)],
        evidence={("B", 0): "s1", ("C", 0): "s2"},
        state_names={("A", 0): [0, 1]},
    )
)

  0%|          | 0/3 [00:00<?, ?it/s]

+-------------+-----------------+
| ('A', 0)    |   phi(('A', 0)) |
+=============+=================+
| ('A', 0)(0) |          0.0000 |
+-------------+-----------------+
| ('A', 0)(1) |          0.0000 |
+-------------+-----------------+



/home/camilo/Repositorios/pgmpy/pgmpy/inference/ApproxInference.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  samples.groupby(variables).size() / samples.shape[0], state_names


In [9]:
# Query the model
dbn_inf = ApproxInference(model)
print(
    dbn_inf.query(
        [("A", 0), ("B", 0)],
        evidence={("C", 0): "s1"},
        state_names={("A", 0): [0, 1], ("B", 0): [0, 1]},
    )
)

  0%|          | 0/3 [00:00<?, ?it/s]
/home/camilo/Repositorios/pgmpy/pgmpy/inference/ApproxInference.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  samples.groupby(variables).size() / samples.shape[0], state_names


+-------------+-------------+--------------------------+
| ('A', 0)    | ('B', 0)    |   phi(('A', 0),('B', 0)) |
+=============+=============+==========================+
| ('A', 0)(0) | ('B', 0)(0) |                   0.0000 |
+-------------+-------------+--------------------------+
| ('A', 0)(0) | ('B', 0)(1) |                   0.0000 |
+-------------+-------------+--------------------------+
| ('A', 0)(1) | ('B', 0)(0) |                   0.0000 |
+-------------+-------------+--------------------------+
| ('A', 0)(1) | ('B', 0)(1) |                   0.0000 |
+-------------+-------------+--------------------------+


In [10]:
# Query the model
dbn_inf = ApproxInference(model)
print(
    dbn_inf.query(
        [("A", 0), ("B", 0)],
        evidence={("C", 0): "s1"},
        state_names={("A", 0): ["s1", "s2"], ("B", 0): ["s1", "s2"]},
    )
)

  0%|          | 0/3 [00:00<?, ?it/s]

+--------------+--------------+--------------------------+
| ('A', 0)     | ('B', 0)     |   phi(('A', 0),('B', 0)) |
+==============+==============+==========================+
| ('A', 0)(s1) | ('B', 0)(s1) |                   0.2509 |
+--------------+--------------+--------------------------+
| ('A', 0)(s1) | ('B', 0)(s2) |                   0.2405 |
+--------------+--------------+--------------------------+
| ('A', 0)(s2) | ('B', 0)(s1) |                   0.2662 |
+--------------+--------------+--------------------------+
| ('A', 0)(s2) | ('B', 0)(s2) |                   0.2424 |
+--------------+--------------+--------------------------+



/home/camilo/Repositorios/pgmpy/pgmpy/inference/ApproxInference.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  samples.groupby(variables).size() / samples.shape[0], state_names


In [11]:
!jupyter nbconvert --to python --stdout dummy-dbn.ipynb > output.txt

[NbConvertApp] Converting notebook dummy-dbn.ipynb to python
